<h2>Description</h2>

Dans ce code, nous allons établir un modèle afin de prédire le débit horaire sur les Champs Élysées.

Imports

In [16]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.inspection import permutation_importance

doc = 'convention.csv'

df_final = pd.read_csv('../datasets_axes_with_all_features/'+ doc, sep=';')

In [17]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9644 entries, 0 to 9643
Columns: 1580 entries, Unnamed: 0 to lag_or_35_23
dtypes: float64(1565), int64(1), object(14)
memory usage: 116.3+ MB


In [18]:
df_final = df_final.copy()
df_final['Date et heure de comptage'] = pd.to_datetime(df_final['Date et heure de comptage'], errors='coerce')
df_final = df_final.sort_values('Date et heure de comptage').reset_index(drop=True)

for col in ['est_vacances', 'est_ferie', 'est_avant_ferie', 'est_pieton']:
    if col in df_final.columns:
        df_final[col] = pd.to_numeric(df_final[col], errors='coerce')

features = [
    'Température', 'precipitations heure',
    'heure_sin', 'heure_cos', 'jour_sin', 'jour_cos', 'mois_sin', 'mois_cos', 
    'force moyenne vent (m/s)', 'jour_semaine', 'est_weekend', 'est_vacances', 'est_avant_vacances',
    'est_ferie', 'est_avant_ferie', 'est_pieton','est_rentree'
]

target = 'Débit horaire'

mask_known   = df_final[target].notna()
mask_missing = df_final[target].isna()

X_known = df_final.loc[mask_known, features].copy()
y_known = df_final.loc[mask_known, target].astype(float)
X_missing = df_final.loc[mask_missing, features].copy()

numeric_features = [
    'Température', 'precipitations heure',
    'heure_sin', 'heure_cos', 'jour_sin', 'jour_cos', 'mois_sin', 'mois_cos',
    'force moyenne vent (m/s)','est_weekend', 'est_vacances', 'est_avant_vacances',
    'est_ferie', 'est_avant_ferie', 'est_pieton', 'est_rentree'
]

categorical_features = ['jour_semaine']

try:
    ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)  
except TypeError:
    ohe = OneHotEncoder(handle_unknown='ignore', sparse=False)         

preprocess = ColumnTransformer(
    transformers=[
        ('num', Pipeline(steps=[('imputer', SimpleImputer(strategy='median'))]), numeric_features),
        ('cat', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('ohe', ohe)
        ]), categorical_features),
    ],
    remainder='drop'
)

model = HistGradientBoostingRegressor(
    loss='absolute_error',   
    max_depth=18,
    max_iter=70,
    early_stopping=False,
    random_state=42
)

pipe = Pipeline(steps=[('prep', preprocess), ('model', model)])

# ---------- 3) Split chronologique ----------
X_train, X_test, y_train, y_test = train_test_split(
    X_known, y_known, test_size=0.2, shuffle=False
)

# ---------- 4) Pondérations (férié / veille / piéton) ----------
W_FERIE   = 3.0
W_AVANT   = 1.5
W_PIETON  = 10
W_RENTREE = 10
POST_SCALE_FERIE = 1.0

def make_weights(X_frame):
    w = np.ones(len(X_frame), dtype=float)
    is_ferie  = X_frame['est_ferie'].fillna(0).astype(int).to_numpy()
    is_avant  = X_frame['est_avant_ferie'].fillna(0).astype(int).to_numpy()
    is_pieton = X_frame['est_pieton'].fillna(0).astype(int).to_numpy()
    is_rentree = X_frame['est_rentree'].fillna(0).astype(int).to_numpy()

    w[is_ferie == 1]  = W_FERIE
    w[is_avant == 1]  = np.maximum(w[is_avant == 1], W_AVANT)
    w[is_pieton == 1] = W_PIETON
    w[is_rentree == 1] = W_RENTREE

    # Option : normalisation pour garder une échelle de perte comparable
    #w *= (len(w) / w.sum())
    return w

w_train = make_weights(X_train)

# ---------- 5) Entraînement ----------
pipe.fit(X_train, y_train, model__sample_weight=w_train)

# ---------- 6) Prédiction + post-ajustement éventuel ----------
y_pred = pipe.predict(X_test)

mask_ferie_test = X_test['est_ferie'].fillna(0).astype(int).to_numpy() == 1
mask_est_pieton_test = X_test['est_pieton'].fillna(0).astype(int).to_numpy() == 1
y_pred[mask_ferie_test] *= POST_SCALE_FERIE
#y_pred[mask_est_pieton_test] *= 0.5

# ---------- 7) Évaluation ----------
r2   = r2_score(y_test, y_pred)
mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"R² : {r2:.3f}")
print(f"MAE : {mae:.2f}")
print(f"RMSE : {rmse:.2f}")

# Diagnostics par sous-régimes
is_pieton_test = X_test['est_pieton'].fillna(0).astype(int) == 1
print(f"Part d'observations piéton (test) : {is_pieton_test.mean():.1%}")
if is_pieton_test.any():
    mae_pieton = mean_absolute_error(y_test[is_pieton_test], y_pred[is_pieton_test])
    print(f"MAE (jours piéton) : {mae_pieton:.2f} (n={is_pieton_test.sum()})")
    mae_non_pieton = mean_absolute_error(y_test[~is_pieton_test], y_pred[~is_pieton_test])
    print(f"MAE (jours non piéton) : {mae_non_pieton:.2f} (n={(~is_pieton_test).sum()})")

df_final['Débit_prédit'] = np.nan
df_final.loc[X_test.index, 'Débit_prédit'] = y_pred


R² : 0.871
MAE : 54.85
RMSE : 76.92
Part d'observations piéton (test) : 0.7%
MAE (jours piéton) : 35.29 (n=7)
MAE (jours non piéton) : 54.99 (n=967)


In [19]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9644 entries, 0 to 9643
Columns: 1581 entries, Unnamed: 0 to Débit_prédit
dtypes: datetime64[ns, UTC](1), float64(1566), int64(1), object(13)
memory usage: 116.3+ MB


In [20]:
from sklearn.inspection import permutation_importance
import pandas as pd
import numpy as np
import plotly.express as px

# 1) Importance par permutation sur le pipeline complet (prétraitements inclus)
perm = permutation_importance(
    estimator=pipe,
    X=X_test,
    y=y_test,
    n_repeats=20,
    random_state=42,
    scoring='neg_root_mean_squared_error'  # cohérent avec votre RMSE
)

imp_df = (
    pd.DataFrame({
        'feature': features,
        'importance_mean': perm.importances_mean,
        'importance_std': perm.importances_std
    })
    .sort_values('importance_mean', ascending=False)
)

print(imp_df.head(20))

# 2) Bar chart Plotly (top 20)
topk = imp_df.head(20).sort_values('importance_mean', ascending=True)
fig = px.bar(
    topk,
    x='importance_mean', y='feature',
    error_x='importance_std',
    orientation='h',
    title='Importance par permutation — Top 20 (plus haut = plus influent)'
)
fig.update_layout(xaxis_title="Perte de performance (Δ RMSE, signe inversé)", yaxis_title="")
fig.show()


                     feature  importance_mean  importance_std
2                  heure_sin       168.989824        3.980441
3                  heure_cos       113.161012        2.844294
4                   jour_sin        19.220800        1.572106
11              est_vacances         5.767191        1.365302
9               jour_semaine         1.618294        0.325458
5                   jour_cos         1.100836        0.215600
10               est_weekend         0.590631        0.136732
13                 est_ferie         0.498423        0.454195
8   force moyenne vent (m/s)         0.151919        0.194909
12        est_avant_vacances         0.000000        0.000000
16               est_rentree         0.000000        0.000000
15                est_pieton        -0.010574        0.012503
1       precipitations heure        -0.018199        0.034585
14           est_avant_ferie        -0.128986        0.043390
0                Température        -0.328747        0.373806
7       

In [21]:
time_index = df_final.loc[X_test.index, 'Date et heure de comptage']

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=time_index,
    y=y_pred,
    mode='lines',
    name='Débit prédit'
))
fig.add_trace(go.Scatter(
    x=time_index,
    y=y_test,
    mode='lines',
    name='Débit réel'
))

fig.update_layout(
    title="Comparaison des débits (réel vs prédit)",
    xaxis_title="Date et heure",
    yaxis_title="Débit horaire (véh/h)",
    hovermode='x unified'
)

fig.show()

In [22]:
X_all = df_final.loc[:, features].copy()

predictions_all = pipe.predict(X_all)

serie = pd.Series(predictions_all, index=df_final.index)

commun = df_final[target].notna() & serie.notna()

predictions_with_know = serie.loc[commun].astype(float)

mae = mean_absolute_error(predictions_with_know, y_known)
rmse = np.sqrt(mean_squared_error(predictions_with_know, y_known))
r2   = r2_score(predictions_with_know, y_known)

print("Nombre de valeurs : " + str(len(predictions_with_know)))
print(f"R² : {r2:.3f}")
print(f"MAE : {mae:.2f}")
print(f"RMSE : {rmse:.2f}")

df_final['débit_prédit_all'] = predictions_all 


Nombre de valeurs : 4867
R² : 0.897
MAE : 44.32
RMSE : 67.25


In [23]:
df_final['débit_prédit_all'].info()

<class 'pandas.core.series.Series'>
RangeIndex: 9644 entries, 0 to 9643
Series name: débit_prédit_all
Non-Null Count  Dtype  
--------------  -----  
9644 non-null   float64
dtypes: float64(1)
memory usage: 75.5 KB


In [24]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_final['Date et heure de comptage'],
    y=predictions_all,
    mode='lines',
    name='Débit prédit'
))
fig.add_trace(go.Scatter(
    x=df_final['Date et heure de comptage'],
    y=df_final['Débit horaire'],
    mode='lines',
    name='Débit réel'
))

fig.update_layout(
    title="Comparaison des débits (réel vs prédit)",
    xaxis_title="Date et heure",
    yaxis_title="Débit horaire (véh/h)",
    hovermode='x unified'
)

fig.show()

<h2>Taux d'occupation</h2>

In [25]:
df_final['debit_hpointe'] = df_final['débit_prédit_all'] * df_final['est_hpointe_soir']

target_occ = 'Taux d\'occupation'

features_occ = [
    'heure_sin', 'heure_cos', 'jour_sin', 'jour_cos', 'mois_sin',  
    'est_weekend', 'est_avant_vacances',
    'est_ferie', 'est_avant_ferie', 'est_pieton', 'lag_or_4_0', 'lag_or_4_1', 'lag_or_5_0', 'lag_or_5_1', 'lag_or_4_23',
    'lag_or_6_23', 'lag_or_7_0', 'lag_or_7_1', 'est_rentree', 'debit_hpointe'
]

mask_occ = df_final[target_occ].notna()
mask_missing_occ = df_final[target_occ].isna()

X_occ = df_final.loc[mask_occ, features_occ].copy()
y_occ = df_final.loc[mask_occ, target_occ].astype(float)

X_train_occ, X_test_occ, y_train_occ, y_test_occ = train_test_split(
    X_occ, y_occ, test_size=0.18, shuffle=False
)

numeric_features_occ = [c for c in features_occ if c != 'jour_semaine']
categorical_features_occ = []

try:
    ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
except TypeError:
    ohe = OneHotEncoder(handle_unknown='ignore', sparse=False)

preprocess_occ = ColumnTransformer(
    transformers=[
        ('num', SimpleImputer(strategy='median'), numeric_features_occ),
        ('cat', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('ohe', ohe)
        ]), categorical_features_occ),
    ],
    remainder='drop'
)

model_occ = HistGradientBoostingRegressor(
    loss='squared_error',
    max_depth=8,
    max_iter=30,
    early_stopping=False,
    random_state=42
)

pipe_occ = Pipeline(steps=[('prep', preprocess_occ), ('model', model_occ)])

pipe_occ.fit(X_train_occ, y_train_occ)

y_pred_occ = pipe_occ.predict(X_test_occ)

print("=== Performances taux d'occupation ===")
print(f"R²   : {r2_score(y_test_occ, y_pred_occ):.3f}")
print(f"MAE  : {mean_absolute_error(y_test_occ, y_pred_occ):.2f}")
print(f"RMSE : {np.sqrt(mean_squared_error(y_test_occ, y_pred_occ)):.2f}")
print(f"nb test : {len(y_test_occ)}")


=== Performances taux d'occupation ===
R²   : 0.537
MAE  : 1.33
RMSE : 2.76
nb test : 896


In [26]:
perm = permutation_importance(
    estimator=pipe_occ,
    X=X_test_occ,
    y=y_test_occ,
    n_repeats=20,
    random_state=42,
    scoring='neg_root_mean_squared_error'  
)

imp_df = (
    pd.DataFrame({
        'feature': features_occ,
        'importance_mean': perm.importances_mean,
        'importance_std': perm.importances_std
    })
    .sort_values('importance_mean', ascending=False)
)

print(imp_df.head(20))

topk = imp_df.head(20).sort_values('importance_mean', ascending=True)
fig = px.bar(
    topk,
    x='importance_mean', y='feature',
    error_x='importance_std',
    orientation='h',
    title='Importance par permutation — Top 20 (plus haut = plus influent)'
)
fig.update_layout(xaxis_title="Perte de performance (Δ RMSE, signe inversé)", yaxis_title="")
fig.show()


               feature  importance_mean  importance_std
16          lag_or_7_0         0.535594        0.033781
0            heure_sin         0.327618        0.024875
19       debit_hpointe         0.214079        0.021415
1            heure_cos         0.080186        0.008510
6   est_avant_vacances         0.058398        0.021671
12          lag_or_5_0         0.048062        0.012591
14         lag_or_4_23         0.045432        0.012412
2             jour_sin         0.043233        0.012824
3             jour_cos         0.024207        0.006656
10          lag_or_4_0         0.021006        0.012137
15         lag_or_6_23         0.016821        0.013752
7            est_ferie         0.005996        0.005481
11          lag_or_4_1         0.001257        0.005632
4             mois_sin         0.000575        0.003399
5          est_weekend         0.000490        0.000288
18         est_rentree         0.000000        0.000000
9           est_pieton        -0.000006        0

In [27]:
time_index_occ = df_final.loc[X_test_occ.index, 'Date et heure de comptage']

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=time_index_occ,
    y=y_pred_occ,
    mode='lines',
    name="Taux d'occupation prédit"
))
fig.add_trace(go.Scatter(
    x=time_index_occ,
    y=y_test_occ,
    mode='lines',
    name= "Taux d'occupation réel"
))

fig.update_layout(
    title="Comparaison des débits (réel vs prédit)",
    xaxis_title="Date et heure",
    yaxis_title="Taux d'occupation (%)",
    hovermode='x unified'
)

fig.show()

In [28]:
X_all_occ = df_final.loc[:, features_occ].copy()

predictions_all_occ = pipe_occ.predict(X_all_occ)

serie_occ = pd.Series(predictions_all_occ, index=df_final.index)

commun_occ = df_final[target_occ].notna() & serie_occ.notna()

predictions_with_know_occ = serie_occ.loc[commun_occ].astype(float)

mae = mean_absolute_error(predictions_with_know_occ, y_occ)
rmse = np.sqrt(mean_squared_error(predictions_with_know_occ, y_occ))
r2   = r2_score(predictions_with_know_occ, y_occ)

print("Nombre de valeurs : " + str(len(predictions_with_know_occ)))
print(f"R² : {r2:.3f}")
print(f"MAE : {mae:.2f}")
print(f"RMSE : {rmse:.2f}")

Nombre de valeurs : 4974
R² : 0.518
MAE : 0.94
RMSE : 1.97


In [29]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_final['Date et heure de comptage'],
    y=predictions_all_occ,
    mode='lines',
    name='Taux prédit'
))
fig.add_trace(go.Scatter(
    x=df_final['Date et heure de comptage'],
    y=df_final[target_occ],
    mode='lines',
    name='Taux d occupation réel'
))

fig.update_layout(
    title="Comparaison des taux occupation (réel vs prédit)",
    xaxis_title="Date et heure",
    yaxis_title="Taux d'occupation (%)",
    hovermode='x unified'
)

fig.show()